# Mall Customer Segmentation Using Unsupervised Learning Algorithms

**Subject:** Unsupervised Learning Algorithms  
**Objective:** Group mall customers based on demographic and spending behaviour using K-Means, Agglomerative Hierarchical Clustering and DBSCAN.

> Run all cells in Google Colab (Runtime → Run all). The dataset is downloaded from the public GitHub repository linked in the report. Internet access is required for the first data-loading cell.


## 1. Import libraries
We use Pandas and NumPy for data handling, Matplotlib/Seaborn for visualisation, and Scikit-learn/SciPy for clustering and evaluation.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import linkage, dendrogram
from IPython.display import display
sns.set_theme(style='whitegrid')


## 2. Load the dataset
The dataset contains 200 customer records and five columns: CustomerID, Gender, Age, Annual Income and Spending Score.


In [ ]:
url = 'https://raw.githubusercontent.com/hannazan/mall-customer-segmentation/main/data/Mall_Customers.csv'
df = pd.read_csv(url)
df.columns = [c.strip() for c in df.columns]
df.rename(columns={'Genre':'Gender'}, inplace=True)
display(df.head())
print('Dataset shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nSummary statistics:')
display(df.describe(include='all'))


## 3. Data cleaning and exploratory data analysis
CustomerID is an identifier, not a behavioural measurement, so it is excluded from clustering. The main segmentation uses Annual Income and Spending Score. Standardisation is important because clustering uses distance calculations and features may have different scales.


In [ ]:
# Remove duplicate records and rows with missing values, if any
df = df.drop_duplicates().dropna().copy()
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['Annual Income (k$)'], kde=True, ax=axes[0])
axes[0].set_title('Annual Income Distribution')
sns.histplot(df['Spending Score (1-100)'], kde=True, ax=axes[1])
axes[1].set_title('Spending Score Distribution')
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x=features[0], y=features[1], hue='Gender', alpha=.75)
plt.title('Mall Customers: Income vs Spending Score')
plt.show()


## 4. K-Means clustering and Elbow Method
K-Means partitions observations into K clusters by minimising the within-cluster sum of squares. The Elbow Method plots inertia for different K values. We use K=5 for the final segmentation, a commonly examined structure for this dataset, and also report the silhouette score.


In [ ]:
inertias = []
k_values = range(2, 11)
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(7,4))
plt.plot(list(k_values), inertias, marker='o')
plt.xlabel('Number of clusters (K)'); plt.ylabel('Inertia')
plt.title('Elbow Method for K-Means'); plt.xticks(list(k_values)); plt.show()

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
df['KMeans_Cluster'] = kmeans_labels
print('K-Means silhouette score:', round(silhouette_score(X_scaled, kmeans_labels), 4))

plt.figure(figsize=(8,6))
sns.scatterplot(x=df[features[0]], y=df[features[1]], hue=kmeans_labels, palette='tab10', s=65)
centres = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centres[:,0], centres[:,1], marker='X', s=220, c='black', label='Centroids')
plt.title('K-Means Customer Segments (K=5)'); plt.legend(title='Cluster'); plt.show()


## 5. Hierarchical Agglomerative Clustering
Agglomerative clustering is a bottom-up method. Each observation begins as its own cluster, and the closest clusters are repeatedly merged. A dendrogram visualises the merging process. We use Ward linkage and five clusters for comparison.


In [ ]:
Z = linkage(X_scaled, method='ward')
plt.figure(figsize=(12,5))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=8)
plt.title('Hierarchical Clustering Dendrogram (Truncated)')
plt.xlabel('Merged observations / cluster size'); plt.ylabel('Ward distance'); plt.show()

hier = AgglomerativeClustering(n_clusters=5, linkage='ward')
hier_labels = hier.fit_predict(X_scaled)
df['Hierarchical_Cluster'] = hier_labels
print('Hierarchical silhouette score:', round(silhouette_score(X_scaled, hier_labels), 4))

plt.figure(figsize=(8,6))
sns.scatterplot(x=df[features[0]], y=df[features[1]], hue=hier_labels, palette='tab10', s=65)
plt.title('Hierarchical Agglomerative Clustering (5 clusters)')
plt.show()


## 6. DBSCAN clustering
DBSCAN is a density-based algorithm. It groups dense regions and marks isolated observations as noise (-1). Unlike K-Means, it does not require the number of clusters in advance. The code tests several epsilon values and chooses the valid setting with the highest silhouette score, while requiring at least two non-noise clusters.


In [ ]:
best_dbscan = None
best_score = -1
dbscan_candidates = []
for eps in np.arange(0.15, 0.81, 0.025):
    candidate = DBSCAN(eps=float(eps), min_samples=5)
    labels = candidate.fit_predict(X_scaled)
    non_noise = labels != -1
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2 and non_noise.sum() > n_clusters:
        score = silhouette_score(X_scaled[non_noise], labels[non_noise])
        dbscan_candidates.append((eps, n_clusters, int((~non_noise).sum()), score))
        if score > best_score:
            best_score = score
            best_dbscan = (candidate, labels, eps)

if best_dbscan is not None:
    dbscan_model, dbscan_labels, chosen_eps = best_dbscan
    df['DBSCAN_Cluster'] = dbscan_labels
    print(f'Selected eps={chosen_eps:.3f}, clusters={len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)}, noise points={(dbscan_labels == -1).sum()}')
    print('DBSCAN silhouette score (excluding noise):', round(best_score, 4))
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=df[features[0]], y=df[features[1]], hue=dbscan_labels, palette='tab10', s=65)
    plt.title('DBSCAN Customer Segments (-1 indicates noise)')
    plt.show()
else:
    dbscan_labels = DBSCAN(eps=0.35, min_samples=5).fit_predict(X_scaled)
    df['DBSCAN_Cluster'] = dbscan_labels
    print('No tested DBSCAN setting produced 2 non-noise clusters. Inspect the plot and parameters; DBSCAN may classify most points as noise or one dense group.')
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=df[features[0]], y=df[features[1]], hue=dbscan_labels, palette='tab10', s=65)
    plt.title('DBSCAN Result (-1 indicates noise)'); plt.show()


## 7. Compare algorithms
Silhouette score ranges from -1 to 1; higher values generally indicate better separation and cohesion. Davies-Bouldin Index is non-negative; lower values generally indicate better separation. For DBSCAN, noise points are excluded from these metrics. Scores should be interpreted alongside the plots and the number of clusters, not used as the only decision criterion.


In [ ]:
def cluster_metrics(Xdata, labels, exclude_noise=False):
    labels = np.asarray(labels)
    mask = labels != -1 if exclude_noise else np.ones(len(labels), dtype=bool)
    X_eval, y_eval = Xdata[mask], labels[mask]
    n_clusters = len(set(y_eval))
    if n_clusters < 2 or n_clusters >= len(y_eval):
        return n_clusters, np.nan, np.nan
    return n_clusters, silhouette_score(X_eval, y_eval), davies_bouldin_score(X_eval, y_eval)

rows = []
for name, labels, exclude in [
    ('K-Means', kmeans_labels, False),
    ('Hierarchical', hier_labels, False),
    ('DBSCAN', dbscan_labels, True)
]:
    n, sil, dbi = cluster_metrics(X_scaled, labels, exclude)
    rows.append({'Algorithm': name, 'Clusters': n, 'Silhouette Score': sil, 'Davies-Bouldin Index': dbi})
comparison = pd.DataFrame(rows)
display(comparison.round(4))

# Save assigned customer segments and metrics for the project submission
df.to_csv('mall_customer_clustered_output.csv', index=False)
comparison.to_csv('algorithm_comparison.csv', index=False)
print('Saved: mall_customer_clustered_output.csv and algorithm_comparison.csv')


## 8. Interpretation and conclusion
K-Means and hierarchical clustering divide customers into a fixed number of groups, making their segments straightforward to compare. DBSCAN identifies dense regions and potential outliers, but its result is sensitive to epsilon and minimum-sample settings. The final choice should consider cluster quality, interpretability and the business question.

For the mall, the segments can support differentiated marketing: retain high-spending customers, encourage moderate spenders with relevant offers, and study low-spending groups to understand barriers to purchase. These are descriptive segments, not proof of customer motivation or future behaviour.

**Submission note:** Include the generated plots and the comparison table from the executed notebook in your report. Do not report numerical scores until the notebook has been run.
